# 02 — Limpeza dos dados do ClinicalTrials.gov
Este notebook transforma os dados brutos (extração de 12/10/2026) em tabelas limpas para análise.


## 1. Ler os dados brutos

In [428]:
import json
import pandas as pd

In [429]:
with open("../data/raw/estudos_obesidade_2026-10-12.json", encoding="utf-8") as ficheiro:
    estudos_brutos = json.load(ficheiro)

In [430]:
len(estudos_brutos)

8431

In [431]:
print(json.dumps(estudos_brutos[0], indent=2, ensure_ascii=False))

{
  "protocolSection": {
    "identificationModule": {
      "nctId": "NCT06151145",
      "briefTitle": "Stepped Care for Weight Loss Maintenance"
    },
    "statusModule": {
      "overallStatus": "ACTIVE_NOT_RECRUITING",
      "startDateStruct": {
        "date": "2024-02-01",
        "type": "ACTUAL"
      },
      "primaryCompletionDateStruct": {
        "date": "2028-06",
        "type": "ESTIMATED"
      },
      "completionDateStruct": {
        "date": "2028-06",
        "type": "ESTIMATED"
      }
    },
    "sponsorCollaboratorsModule": {
      "leadSponsor": {
        "class": "OTHER"
      }
    },
    "designModule": {
      "studyType": "INTERVENTIONAL",
      "phases": [
        "NA"
      ],
      "enrollmentInfo": {
        "count": 192
      }
    },
    "armsInterventionsModule": {
      "interventions": [
        {
          "type": "BEHAVIORAL"
        },
        {
          "type": "BEHAVIORAL"
        },
        {
          "type": "BEHAVIORAL"
        }
      

In [432]:
print(json.dumps(estudos_brutos[0]["protocolSection"]["designModule"], indent=2))

{
  "studyType": "INTERVENTIONAL",
  "phases": [
    "NA"
  ],
  "enrollmentInfo": {
    "count": 192
  }
}


In [433]:
for estudo in estudos_brutos:
    status = estudo["protocolSection"]["statusModule"]
    if status["overallStatus"] == "TERMINATED":
        print(json.dumps(status, indent=2))
        break

{
  "overallStatus": "TERMINATED",
  "whyStopped": "difficulty in recruiting",
  "startDateStruct": {
    "date": "2013-12",
    "type": "ACTUAL"
  },
  "primaryCompletionDateStruct": {
    "date": "2022-01",
    "type": "ACTUAL"
  },
  "completionDateStruct": {
    "date": "2022-06",
    "type": "ACTUAL"
  }
}


## 2. Construir a tabela estudos

In [434]:
linhas = []
for estudo in estudos_brutos:
   
    protocolo = estudo.get("protocolSection", {})

    
    ident = protocolo.get("identificationModule", {})
    status =protocolo.get("statusModule",{})
    design = protocolo.get("designModule",{})
    patrocinio = protocolo.get("sponsorCollaboratorsModule",{})

    
    inicio = status.get("startDateStruct", {})
    fim_primaria = status.get("primaryCompletionDateStruct",{})
    fim = status.get("completionDateStruct",{})
    inscricao = design.get("enrollmentInfo",{})
    patrocinador = patrocinio.get("leadSponsor",{})

   
    fases = design.get("phases",[])

    linhas.append({
        "nct_id": ident.get("nctId"),
        "titulo": ident.get("briefTitle"),
        "tipo_estudo": design.get("studyType"),
        "estado": status.get("overallStatus"),
        "motivo_paragem": status.get("whyStopped"),
        "data_inicio": inicio.get("date"),
        "tipo_data_inicio": inicio.get("type"),
        "data_fim_primaria": fim_primaria.get("date"),
        "tipo_data_fim_primaria": fim_primaria.get("type"),
        "data_fim": fim.get("date"),
        "tipo_data_fim": fim.get("type"),
        "participantes": inscricao.get("count"),
        "fase": "/".join(fases) if fases else None,
        "classe_patrocinador": patrocinador.get("class"),
    })

df_estudos = pd.DataFrame(linhas)

In [435]:
df_estudos.shape

(8431, 14)

In [436]:
df_estudos.head()

,nct_id,titulo,tipo_estudo,estado,motivo_paragem,data_inicio,tipo_data_inicio,data_fim_primaria,tipo_data_fim_primaria,data_fim,tipo_data_fim,participantes,fase,classe_patrocinador
0,NCT06151145,Stepped Care for Weight Loss Maintenance,INTERVENTIONAL,ACTIVE_NOT_RECRUITING,NaN,2024-02-01,ACTUAL,2028-06,ESTIMATED,2028-06,ESTIMATED,192.0,NA,OTHER
1,NCT02185872,CrossFit Exercise to Improve Glucose Control for Overweight and Obese Adults,INTERVENTIONAL,COMPLETED,NaN,2012-02,NaN,2012-05,ACTUAL,2012-05,ACTUAL,23.0,NA,OTHER
2,NCT03917901,Computer-Delivered Intervention for Individuals With Obesity and Elevated Anxiety Sensitivity,INTERVENTIONAL,COMPLETED,NaN,2019-09-16,ACTUAL,2021-03-06,ACTUAL,2021-03-06,ACTUAL,131.0,NA,OTHER
3,NCT01049698,Improving Muscle for Functional Independence Trial,INTERVENTIONAL,COMPLETED,NaN,2010-02,NaN,2014-05,ACTUAL,2014-05,ACTUAL,126.0,NA,OTHER
4,NCT06989177,Probiotic Intervention on Body Weight,INTERVENTIONAL,RECRUITING,NaN,2025-06-10,ACTUAL,2026-12-30,ESTIMATED,2026-12-31,ESTIMATED,140.0,NA,OTHER_GOV


In [437]:
df_estudos.isna().sum()

nct_id                       0
titulo                       0
tipo_estudo                  0
estado                       0
motivo_paragem            7935
data_inicio                  0
tipo_data_inicio          2062
data_fim_primaria            3
tipo_data_fim_primaria       4
data_fim                    54
tipo_data_fim               57
participantes                2
fase                         3
classe_patrocinador          0
dtype: int64

In [438]:
parados = df_estudos["estado"].isin(["TERMINATED", "SUSPENDED", "WITHDRAWN"])
df_estudos[parados]["motivo_paragem"].isna().sum()

np.int64(40)

In [439]:
df_estudos["nct_id"].is_unique


True

In [440]:
df_estudos["tipo_estudo"].value_counts(dropna=False)

tipo_estudo
INTERVENTIONAL    8431
Name: count, dtype: int64

In [441]:
sem_tipo = df_estudos["tipo_data_inicio"].isna()
df_estudos[sem_tipo]["data_inicio"].str[:4].value_counts().sort_index()

data_inicio
2010    325
2011    317
2012    353
2013    323
2014    280
2015    263
2016    193
2017      8
Name: count, dtype: int64

In [442]:
df_estudos["tipo_data_inicio"].isna().mean()

np.float64(0.24457359743802634)

In [443]:
parados.sum()

np.int64(536)

In [444]:
df_estudos[~parados]["motivo_paragem"].notna().sum()

np.int64(0)

In [445]:
(df_estudos["data_fim"].notna() & df_estudos["tipo_data_fim"].isna()).sum()

np.int64(3)

In [446]:
df_estudos["participantes"]= df_estudos["participantes"].astype("Int64")

In [447]:
df_estudos.info()

<class 'pandas.DataFrame'>
RangeIndex: 8431 entries, 0 to 8430
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   nct_id                  8431 non-null   str  
 1   titulo                  8431 non-null   str  
 2   tipo_estudo             8431 non-null   str  
 3   estado                  8431 non-null   str  
 4   motivo_paragem          496 non-null    str  
 5   data_inicio             8431 non-null   str  
 6   tipo_data_inicio        6369 non-null   str  
 7   data_fim_primaria       8428 non-null   str  
 8   tipo_data_fim_primaria  8427 non-null   str  
 9   data_fim                8377 non-null   str  
 10  tipo_data_fim           8374 non-null   str  
 11  participantes           8429 non-null   Int64
 12  fase                    8428 non-null   str  
 13  classe_patrocinador     8431 non-null   str  
dtypes: Int64(1), str(13)
memory usage: 930.5 KB


### Observações da inspeção

- **Dimensão:** a tabela tem 8 431 estudos e 14 colunas.
- **Validações:** o `nct_id` é único (não há estudos repetidos) e os 8 431 estudos são todos INTERVENTIONAL, o que confirma o filtro da extração.
- **Estudos parados sem motivo:** dos 536 estudos parados, 40 não indicam o motivo. Não há estudos não parados com motivo.
- **Tipo da data de início:** 24,5 % dos estudos não têm tipo (real ou estimada). Todos começaram entre 2010 e 2017; a partir de 2018 não há nenhum caso, o que sugere uma mudança na forma como os dados eram recolhidos.
- **Data de fim:** 54 estudos não têm data de fim. Outros 3 têm a data, mas não indicam se é real ou estimada.
- **Data de fim primária:** 3 estudos não têm esta data. Outro 1 tem a data, mas não indica se é real ou estimada.
- **Participantes e fase:** 2 estudos não indicam o número de participantes e 3 não indicam a fase.

## 3. Construir a tabela estudo_pais


In [448]:
linhas_pais = []
for estudo in estudos_brutos:                      
    protocolo = estudo.get("protocolSection", {})
    ident = protocolo.get("identificationModule", {})
    locais = protocolo.get("contactsLocationsModule", {})
    for centro in locais.get("locations", []):           
        linhas_pais.append({
            "nct_id": ident.get("nctId"),
            "pais": centro.get("country"),
        })

df_estudo_pais = pd.DataFrame(linhas_pais)
df_estudo_pais.shape

(29961, 2)

In [449]:
df_estudo_pais= df_estudo_pais.drop_duplicates().reset_index(drop=True)
df_estudo_pais.shape

(9270, 2)

In [450]:
df_estudo_pais["nct_id"].nunique()


7837

In [451]:
df_estudo_pais.isna().sum()

nct_id    0
pais      0
dtype: int64

In [452]:
df_estudo_pais["pais"].nunique()

107

In [453]:
print(df_estudo_pais["pais"].value_counts().sort_index().to_string())

pais
Albania                      1
Algeria                      1
Argentina                   55
Australia                  114
Austria                     46
Bangladesh                   4
Belgium                     84
Bolivia                      1
Bosnia and Herzegovina       1
Brazil                     211
Brunei                       1
Bulgaria                    23
Cameroon                     1
Canada                     407
Chile                       42
China                      419
Colombia                    20
Costa Rica                   1
Croatia                     14
Cuba                         2
Cyprus                       9
Czechia                     73
Denmark                    228
Ecuador                      2
Egypt                      161
Estonia                      3
Ethiopia                     2
Finland                     68
France                     341
Georgia                      6
Germany                    213
Greece                      72
Gre

In [454]:
total_estudos = df_estudos["nct_id"].nunique()
estudos_com_pais = df_estudo_pais["nct_id"].nunique()
estudos_sem_pais = total_estudos - estudos_com_pais
round(estudos_sem_pais / total_estudos * 100, 1)

7.0

In [455]:
sem_pais = ~df_estudos["nct_id"].isin(df_estudo_pais["nct_id"])
print(sem_pais.sum())
print(round(sem_pais.mean() * 100, 1))

594
7.0


In [456]:
df_estudo_pais["nct_id"].isin(df_estudos["nct_id"]).all()

np.True_

In [457]:
(df_estudo_pais["pais"] == "Portugal").sum()

np.int64(53)

### Observações da tabela estudo_pais

- **Dimensão:** 29 961 linhas (centros) antes de remover duplicados;9 270 pares estudo–país depois.
- **Estudos sem país:** 594 estudos (7 %) não indicam nenhum país.
- **Países:** 107 países diferentes. Não foram encontradas variantes de escrita.
- **Decisões:** 1 estudo indica "Serbia and Montenegro", nome anterior a 2006; mantido tal como registado. Os territórios (ex.: Puerto Rico, Hong Kong) foram mantidos separados, tal como no ClinicalTrials.gov.
- **Integridade:** todos os `nct_id` desta tabela existem na tabela `estudos`.
- **Portugal:** Participa em 53 estudos.

## 4. Construir a tabela estudo_intervencao

In [458]:
linhas_intervencao = []
for estudo in estudos_brutos:                      
    protocolo = estudo.get("protocolSection", {})
    ident = protocolo.get("identificationModule", {})
    modulo_interv = protocolo.get("armsInterventionsModule", {})
    for intervencao in modulo_interv.get("interventions", []):           
        linhas_intervencao.append({
            "nct_id": ident.get("nctId"),
            "tipo_intervencao": intervencao.get("type"),
        })

df_estudo_intervencao = pd.DataFrame(linhas_intervencao)
df_estudo_intervencao.shape

(16524, 2)

In [459]:
df_estudo_intervencao.isna().sum()

nct_id              0
tipo_intervencao    0
dtype: int64

In [460]:
df_estudo_intervencao["tipo_intervencao"].value_counts(dropna=False)

tipo_intervencao
BEHAVIORAL             5172
DRUG                   3607
OTHER                  3588
DIETARY_SUPPLEMENT     1974
DEVICE                  945
PROCEDURE               882
DIAGNOSTIC_TEST         150
BIOLOGICAL              110
COMBINATION_PRODUCT      66
GENETIC                  16
RADIATION                14
Name: count, dtype: int64

In [461]:
df_estudo_intervencao = df_estudo_intervencao.drop_duplicates().reset_index(drop=True)

In [462]:
type(df_estudo_intervencao)

pandas.DataFrame

In [463]:

df_estudo_intervencao["nct_id"].isin(df_estudos["nct_id"]).all()

np.True_

In [464]:
df_estudo_intervencao.shape

(9697, 2)

In [465]:
df_estudo_intervencao["tipo_intervencao"].value_counts()

tipo_intervencao
BEHAVIORAL             3107
OTHER                  2173
DRUG                   1790
DIETARY_SUPPLEMENT     1141
DEVICE                  683
PROCEDURE               590
DIAGNOSTIC_TEST          75
BIOLOGICAL               63
COMBINATION_PRODUCT      52
GENETIC                  13
RADIATION                10
Name: count, dtype: int64

In [466]:
sem_intervencao= ~df_estudos["nct_id"].isin(df_estudo_intervencao["nct_id"])


In [467]:
print(sem_intervencao.sum())
print(round(sem_intervencao.mean()*100,1))

0
0.0


In [468]:
df_estudo_intervencao["tipo_intervencao"].nunique() 

11

### Observações da tabela estudo_intervencao
- **Dimensão:** 16 524 linhas (intervenções) antes de remover duplicados; 9697 pares estudo–tipo depois.
- **Duplicados:** a maior descida foi no tipo DRUG (−50 %), o que é compatível com ensaios que registam o medicamento e o placebo, ambos como DRUG.
- **Estudos sem intervenção:** 0 (todos os estudos intervencionais indicam pelo menos uma intervenção).
- **Tipos:** 11 tipos, já padronizados pelo ClinicalTrials.gov; sem variantes de escrita.
- **Limitação:** o tipo OTHER (2 173 estudos) agrupa intervenções diversas e pouco específicas.
- **Nota para a análise:** um estudo pode ter vários tipos, por isso as contagens por tipo somam 9 697(mais do que os 8 431 estudos) e as percentagens por tipo não somam 100 %.
- **Integridade:** todos os `nct_id` desta tabela existem na tabela `estudos`.

## 5. Converter as datas

In [469]:
for coluna in ["data_inicio","data_fim_primaria", "data_fim"]:
    print(df_estudos[coluna].str.len().value_counts(dropna=False))
    print(df_estudos[coluna].str.contains("/", na=False).sum())
    print("----")

data_inicio
10    5765
7     2666
Name: count, dtype: int64
0
----
data_fim_primaria
10.0    5507
7.0     2921
NaN        3
Name: count, dtype: int64
0
----
data_fim
10.0    5484
7.0     2893
NaN       54
Name: count, dtype: int64
0
----


In [470]:
df_estudos.columns

Index(['nct_id', 'titulo', 'tipo_estudo', 'estado', 'motivo_paragem',
       'data_inicio', 'tipo_data_inicio', 'data_fim_primaria',
       'tipo_data_fim_primaria', 'data_fim', 'tipo_data_fim', 'participantes',
       'fase', 'classe_patrocinador'],
      dtype='str')

In [471]:
for coluna in ["data_inicio","data_fim_primaria", "data_fim"]:
    df_estudos[f"precisao_{coluna}"]=df_estudos[coluna].str.len().map({10:"dia",7:"mes"})
    so_mes=df_estudos[coluna].str.len()==7
    vazios_antes = df_estudos[coluna].isna().sum()  
    df_estudos.loc[so_mes,coluna]= df_estudos.loc[so_mes,coluna] + "-01"
    df_estudos[coluna]=pd.to_datetime(df_estudos[coluna],format="%Y-%m-%d")
    vazios_depois = df_estudos[coluna].isna().sum()
    print(coluna, vazios_antes, vazios_depois)


data_inicio 0 0
data_fim_primaria 3 3
data_fim 54 54


In [472]:
df_estudos.dtypes

nct_id                                   str
titulo                                   str
tipo_estudo                              str
estado                                   str
motivo_paragem                           str
data_inicio                   datetime64[us]
tipo_data_inicio                         str
data_fim_primaria             datetime64[us]
tipo_data_fim_primaria                   str
data_fim                      datetime64[us]
tipo_data_fim                            str
participantes                          Int64
fase                                     str
classe_patrocinador                      str
precisao_data_inicio                     str
precisao_data_fim_primaria               str
precisao_data_fim                        str
dtype: object

In [473]:
df_estudos["data_inicio"].min()

Timestamp('2010-01-01 00:00:00')

In [474]:
df_estudos["data_inicio"].max()

Timestamp('2025-12-31 00:00:00')

In [475]:
df_estudos["precisao_data_inicio"].value_counts(dropna=False)

precisao_data_inicio
dia    5765
mes    2666
Name: count, dtype: int64

In [476]:
df_estudos["ano_inicio"]= df_estudos["data_inicio"].dt.year

In [477]:
df_estudos["ano_inicio"]

0       2024
1       2012
2       2019
3       2010
4       2025
        ... 
8426    2024
8427    2011
8428    2024
8429    2014
8430    2021
Name: ano_inicio, Length: 8431, dtype: int32

In [478]:
df_estudos.dtypes

nct_id                                   str
titulo                                   str
tipo_estudo                              str
estado                                   str
motivo_paragem                           str
data_inicio                   datetime64[us]
tipo_data_inicio                         str
data_fim_primaria             datetime64[us]
tipo_data_fim_primaria                   str
data_fim                      datetime64[us]
tipo_data_fim                            str
participantes                          Int64
fase                                     str
classe_patrocinador                      str
precisao_data_inicio                     str
precisao_data_fim_primaria               str
precisao_data_fim                        str
ano_inicio                             int32
dtype: object

In [479]:
df_estudos["ano_inicio"].value_counts().sort_index()

ano_inicio
2010    359
2011    388
2012    441
2013    431
2014    437
2015    498
2016    519
2017    547
2018    553
2019    538
2020    436
2021    554
2022    580
2023    687
2024    702
2025    761
Name: count, dtype: int64

In [480]:
round((df_estudos["precisao_data_inicio"] == "mes").mean() * 100, 1)

np.float64(31.6)

In [481]:
round((df_estudos["precisao_data_fim_primaria"] == "mes").mean() * 100, 1)

np.float64(34.6)

In [482]:
round((df_estudos["precisao_data_fim"] == "mes").mean() * 100, 1)

np.float64(34.3)

### Observações da conversão das datas
- **Formatos:** só existem dois formatos (AAAA-MM-DD e AAAA-MM); não há datas com barras.
- **Datas só com mês:** completadas com o dia 1. A precisão original ficou guardada nas colunas `precisao_...`: início 31,6 %, fim primário 34,6 %, fim 34,3 %.
- **Conversão sem perdas:** os vazios antes e depois da conversão são iguais (0, 3 e 54).
- **Validação:** as datas de início estão entre 1 de janeiro de 2010 e 31 dezembro de 2025, o que confirma o filtro da extração.
- **Nova coluna:** `ano_inicio` (número inteiro).
- **Atenção na análise:** as datas com precisão "mes" não devem ser usadas em análises ao nível do dia (por exemplo, dia da semana).

## 6. Calcular a duração dos ensaios

In [483]:
df_estudos["tipo_data_inicio"].value_counts(dropna=False)

tipo_data_inicio
ACTUAL       5933
NaN          2062
ESTIMATED     436
Name: count, dtype: int64

In [484]:
df_estudos["tipo_data_fim_primaria"].value_counts(dropna=False)

tipo_data_fim_primaria
ACTUAL       6097
ESTIMATED    2330
NaN             4
Name: count, dtype: int64

In [485]:
DATA_EXTRACAO = pd.Timestamp("2026-10-12")

In [486]:
df_estudos[df_estudos["tipo_data_inicio"] == "ESTIMATED"]["estado"].value_counts()

estado
UNKNOWN                    224
NOT_YET_RECRUITING         121
WITHDRAWN                   68
RECRUITING                  19
SUSPENDED                    2
ENROLLING_BY_INVITATION      2
Name: count, dtype: int64

In [487]:
df_estudos[df_estudos["tipo_data_inicio"] == "ACTUAL"]["estado"].value_counts(dropna=False)


estado
COMPLETED                  3823
RECRUITING                  707
UNKNOWN                     599
ACTIVE_NOT_RECRUITING       426
TERMINATED                  242
WITHDRAWN                    73
ENROLLING_BY_INVITATION      49
SUSPENDED                    14
Name: count, dtype: int64

In [488]:
fim_real= df_estudos["tipo_data_fim_primaria"]=="ACTUAL"
fim_real.sum()

np.int64(6097)

In [489]:
inicio_real= df_estudos["tipo_data_inicio"]=="ACTUAL"
inicio_real.sum()

np.int64(5933)

In [490]:
fim_no_futuro= fim_real & (df_estudos["data_fim_primaria"] > DATA_EXTRACAO)
fim_no_futuro.sum()

np.int64(0)

In [491]:
fim_antes_inicio= df_estudos["data_fim_primaria"]< df_estudos["data_inicio"]
fim_antes_inicio.sum()

np.int64(0)

In [492]:
concluido_estimado= (df_estudos["estado"]=="COMPLETED") & (df_estudos["tipo_data_fim_primaria"]=="ESTIMATED")
concluido_estimado.sum()

np.int64(0)

In [493]:
#datas (sem olhar ao estado)
datas_ok = (fim_real & df_estudos["data_inicio"].notna()
            & ~fim_no_futuro & ~fim_antes_inicio)
datas_ok.sum()

np.int64(6097)

In [494]:
df_estudos[datas_ok]["estado"].value_counts()

estado
COMPLETED                  5430
TERMINATED                  317
ACTIVE_NOT_RECRUITING       150
UNKNOWN                      94
WITHDRAWN                    75
RECRUITING                   19
ENROLLING_BY_INVITATION       8
SUSPENDED                     4
Name: count, dtype: int64

In [495]:
chegou_resultado = df_estudos["estado"].isin(["COMPLETED", "ACTIVE_NOT_RECRUITING", "UNKNOWN"])
valida_pragmatica = datas_ok & chegou_resultado
valida_estrita = datas_ok & inicio_real & chegou_resultado

In [496]:
valida_pragmatica.sum()


np.int64(5674)

In [497]:
valida_estrita.sum()

np.int64(4033)

In [498]:
df_estudos["duracao_meses"] = ((df_estudos["data_fim_primaria"] - df_estudos["data_inicio"]).dt.days / 30.44).round(1)

In [499]:
df_estudos.loc[valida_pragmatica, "duracao_meses"].median()

np.float64(17.0)

In [500]:
df_estudos.loc[valida_estrita, "duracao_meses"].median()

np.float64(16.2)

In [501]:
df_estudos.loc[valida_pragmatica, "duracao_meses"].mean()

np.float64(22.743214663376808)

In [502]:
df_estudos.loc[~valida_pragmatica, "duracao_meses"] = None

In [503]:
df_estudos["duracao_meses"].describe()

count    5674.000000
mean       22.743215
std        19.593709
min         0.000000
25%         8.000000
50%        17.000000
75%        31.200000
max       188.200000
Name: duracao_meses, dtype: float64

In [504]:
(df_estudos["duracao_meses"] == 0).sum()

np.int64(20)

In [505]:
df_estudos.sort_values("duracao_meses", ascending=False)[["titulo", "duracao_meses"]].head()

,titulo,duracao_meses
2957,Role of Immune System in Obesity-related Inflammation and Cardiometabolic Risk,188.2
5449,The Expecting Study,140.9
2773,An Exercise RCT Targeting African-American Women With Metabolic Syndrome and High Risk for Breast Cancer,140.0
3479,Starting Early Obesity Prevention Program,136.0
1946,An Endometrial Cancer Chemoprevention Study of Metformin,132.0


In [506]:
zero = df_estudos["duracao_meses"] == 0
df_estudos[zero][["precisao_data_inicio", "precisao_data_fim_primaria"]].value_counts()

precisao_data_inicio  precisao_data_fim_primaria
mes                   mes                           11
dia                   dia                            9
Name: count, dtype: int64

In [507]:
df_estudos[zero]["titulo"]

1986                                                                                                     Inflammation Regulation in Obese Patients
2132    The Effect of Web-Based and Face-to-face Training Given to Office Workers on Health Beliefs and Physical Activity Levels Regarding Obesity
2360                                         Mobile Health (M-health) Intervention to Reduce the Epigenetic Signature in Metabolic Syndrome (MetS)
2997                                                                                 Mindfulness Intervention for Overweight Primary Care Patients
3369                                                                                               Swimming and Water Walking on Spirometry Values
3577                                                                                           Effects of Sugary Drinks Counter-marketing Messages
3760         Combined Training (Strength and Endurance) and the Serum Levels of Myostatin and Hormone-sensitive Lipase

### Observações da duração dos ensaios

- **Data de fim usada:** fim primário (tempo até ao resultado principal).
- **Tipos das datas:** início com 5 933 ACTUAL, 436 ESTIMATED e 2 062 sem tipo (todos de 2010 a 2017). Fim primário com 6 097 ACTUAL, 2 330 ESTIMATED e 4 sem tipo.
- **Validações:** 0 fins reais posteriores à extração (12/10/2026), 0 fins anteriores ao início e 0 estudos COMPLETED com o fim primário estimado.
- **Inconsistências:** 2 estudos com o início ESTIMATED mas o fim primário ACTUAL; 27 estudos ainda a recrutar (RECRUITING ou ENROLLING_BY_INVITATION) com o fim primário ACTUAL (excluídos).
- **Estudos incluídos:** fim primário ACTUAL, início com data e estado COMPLETED, ACTIVE_NOT_RECRUITING ou UNKNOWN (5 674 estudos). Os estudos interrompidos (TERMINATED, SUSPENDED, WITHDRAWN) foram excluídos, porque não chegaram ao resultado principal.
- **Resultado e sensibilidade:** mediana de 17,0 meses. Usando só inícios ACTUAL (4 033 estudos), a mediana é de 16,2 meses: o resultado é robusto à escolha do critério.
- **Distribuição:** metade dos ensaios demora entre 8,0 e 31,2 meses. A média (22,7 meses) é puxada por alguns ensaios muito longos (máximo de 188,2 meses).
- **Durações de 0 meses:** 20 estudos (11 com as duas datas só com mês e 9 com dia). Incluem estudos agudos (ex.: biodisponibilidade) e limitações de registo. Mantidos; impacto desprezável na mediana.

## 7. Categorias


In [508]:
df_estudos["estado"].value_counts(dropna=False)


estado
COMPLETED                  5430
UNKNOWN                    1129
RECRUITING                  728
ACTIVE_NOT_RECRUITING       434
TERMINATED                  319
WITHDRAWN                   198
NOT_YET_RECRUITING          121
ENROLLING_BY_INVITATION      53
SUSPENDED                    19
Name: count, dtype: int64

In [509]:
grupo_estado={
    "COMPLETED":"Concluído",
    "RECRUITING" :"Em curso",
    "ACTIVE_NOT_RECRUITING":"Em curso",
    "ENROLLING_BY_INVITATION":"Em curso",
    "NOT_YET_RECRUITING":"Por iniciar",
    "WITHDRAWN":"Retirado",
    "TERMINATED":"Interrompido",
    "SUSPENDED":"Interrompido",
    "UNKNOWN":"Desconhecido",
    }

df_estudos["grupo_estado"]= df_estudos["estado"].map(grupo_estado)    

In [510]:
df_estudos["grupo_estado"].isna().sum()

np.int64(0)

In [511]:
df_estudos["estado"].isna().sum()

np.int64(0)

In [512]:
df_estudos["grupo_estado"].value_counts(dropna=False)

grupo_estado
Concluído       5430
Em curso        1215
Desconhecido    1129
Interrompido     338
Retirado         198
Por iniciar      121
Name: count, dtype: int64

In [513]:
pd.crosstab(df_estudos["estado"],df_estudos["grupo_estado"], dropna=False)

grupo_estado,Concluído,Desconhecido,Em curso,Interrompido,Por iniciar,Retirado
estado,,,,,,
ACTIVE_NOT_RECRUITING,0,0,434,0,0,0
COMPLETED,5430,0,0,0,0,0
ENROLLING_BY_INVITATION,0,0,53,0,0,0
NOT_YET_RECRUITING,0,0,0,0,121,0
RECRUITING,0,0,728,0,0,0
SUSPENDED,0,0,0,19,0,0
TERMINATED,0,0,0,319,0,0
UNKNOWN,0,1129,0,0,0,0
WITHDRAWN,0,0,0,0,0,198


### Observações das categorias

- **Estado:** os 9 códigos foram agrupados em 6 grupos: Concluído (COMPLETED), Em curso (RECRUITING, ACTIVE_NOT_RECRUITING, ENROLLING_BY_INVITATION), Por iniciar (NOT_YET_RECRUITING), Interrompido (TERMINATED, SUSPENDED), Retirado (WITHDRAWN) e Desconhecido (UNKNOWN: o registo não é atualizado há mais de 2 anos). Nenhum código ficou sem grupo.

In [514]:
df_estudos["fase"].value_counts(dropna=False)

fase
NA               6486
PHASE1            456
PHASE2            447
PHASE4            423
PHASE3            319
PHASE1/PHASE2     113
EARLY_PHASE1       99
PHASE2/PHASE3      85
NaN                 3
Name: count, dtype: int64

In [515]:
grupo_fase={
    "EARLY_PHASE1": "Fase 1 precoce",
    "PHASE1": "Fase 1",
    "PHASE1/PHASE2": "Fase 1/2",
    "PHASE2": "Fase 2",
    "PHASE2/PHASE3": "Fase 2/3",
    "PHASE3": "Fase 3",
    "PHASE4": "Fase 4",
    "NA": "Não aplicável",
}

df_estudos["grupo_fase"]= df_estudos["fase"].map(grupo_fase)

In [516]:
df_estudos["grupo_fase"].isna().sum()

np.int64(3)

In [517]:
df_estudos["fase"].isna().sum()

np.int64(3)

In [518]:
df_estudos["grupo_fase"]=df_estudos["grupo_fase"].fillna("Não indicado")

In [519]:
pd.crosstab(df_estudos["fase"],df_estudos["grupo_fase"], dropna=False)

grupo_fase,Fase 1,Fase 1 precoce,Fase 1/2,Fase 2,Fase 2/3,Fase 3,Fase 4,Não aplicável,Não indicado
fase,,,,,,,,,
EARLY_PHASE1,0,99,0,0,0,0,0,0,0
NA,0,0,0,0,0,0,0,6486,0
PHASE1,456,0,0,0,0,0,0,0,0
PHASE1/PHASE2,0,0,113,0,0,0,0,0,0
PHASE2,0,0,0,447,0,0,0,0,0
PHASE2/PHASE3,0,0,0,0,85,0,0,0,0
PHASE3,0,0,0,0,0,319,0,0,0
PHASE4,0,0,0,0,0,0,423,0,0
NaN,0,0,0,0,0,0,0,0,3



- **Fase:** os códigos foram traduzidos para português (ex.: PHASE2/PHASE3 → Fase 2/3; EARLY_PHASE1 → Fase 1 precoce). O código "NA" (6 486 estudos, sobretudo de dieta, exercício e comportamento, que não têm fases) passou a "Não aplicável", antes de gravar o CSV, para não ser lido como vazio. Os 3 estudos sem fase registada ficaram como "Não indicado".


In [520]:
df_estudos["classe_patrocinador"].value_counts(dropna=False)

classe_patrocinador
OTHER        6979
INDUSTRY     1055
OTHER_GOV     249
FED            94
NIH            37
NETWORK         9
INDIV           8
Name: count, dtype: int64

In [521]:
grupo_patrocinador={
    "NIH":"Governo",
    "FED":"Governo",
    "OTHER_GOV": "Governo",
    "INDUSTRY":"Indústria",
    "OTHER":"Universidades, hospitais e outros",
    "NETWORK":"Universidades, hospitais e outros",
    "INDIV":"Universidades, hospitais e outros",
}

df_estudos["grupo_patrocinador"]= df_estudos["classe_patrocinador"].map(grupo_patrocinador)

In [522]:
df_estudos["grupo_patrocinador"].isna().sum()

np.int64(0)

In [523]:
df_estudos["classe_patrocinador"].isna().sum()

np.int64(0)

In [524]:
pd.crosstab(df_estudos["classe_patrocinador"],df_estudos["grupo_patrocinador"], dropna=False)

grupo_patrocinador,Governo,Indústria,"Universidades, hospitais e outros"
classe_patrocinador,,,
FED,94,0,0
INDIV,0,0,8
INDUSTRY,0,1055,0
NETWORK,0,0,9
NIH,37,0,0
OTHER,0,0,6979
OTHER_GOV,249,0,0


In [525]:
df_estudos["grupo_patrocinador"].value_counts(dropna=False)

grupo_patrocinador
Universidades, hospitais e outros    6996
Indústria                            1055
Governo                               380
Name: count, dtype: int64

- **Patrocinador:** os 7 códigos foram agrupados em 3 grupos: Universidades, hospitais e outros (OTHER, NETWORK e INDIV; 6 996 estudos), Indústria (INDUSTRY; 1 055) e Governo (NIH, FED e OTHER_GOV; 380). O NETWORK (redes de investigação, 9 estudos) e o INDIV (investigador individual, 8 estudos) foram juntos ao OTHER por serem muito pequenos e por estarem normalmente ligados a universidades ou hospitais.

In [526]:
df_estudo_intervencao["tipo_intervencao"].value_counts(dropna=False)

tipo_intervencao
BEHAVIORAL             3107
OTHER                  2173
DRUG                   1790
DIETARY_SUPPLEMENT     1141
DEVICE                  683
PROCEDURE               590
DIAGNOSTIC_TEST          75
BIOLOGICAL               63
COMBINATION_PRODUCT      52
GENETIC                  13
RADIATION                10
Name: count, dtype: int64

In [527]:
grupo_intervencao={
    "BEHAVIORAL":"Comportamental",
    "OTHER":"Outro",
    "DRUG":"Medicamento",
    "DIETARY_SUPPLEMENT":"Suplemento alimentar",
    "DEVICE":"Dispositivo médico",
    "PROCEDURE":"Procedimento",
    "DIAGNOSTIC_TEST":"Teste diagnóstico",
    "BIOLOGICAL":"Produto biológico",
    "COMBINATION_PRODUCT": "Produto combinado",
    "GENETIC": "Genético",
    "RADIATION":"Radiação",
}

df_estudo_intervencao["grupo_intervencao"]=df_estudo_intervencao["tipo_intervencao"].map(traducao_intervencao)

In [528]:
df_estudo_intervencao["grupo_intervencao"].isna().sum()

np.int64(0)

In [529]:
df_estudo_intervencao["tipo_intervencao"].isna().sum()

np.int64(0)

In [530]:
pd.crosstab(df_estudo_intervencao["tipo_intervencao"],df_estudo_intervencao["grupo_intervencao"], dropna=False)

grupo_intervencao,Comportamental,Dispositivo médico,Genético,Medicamento,Outro,Procedimento,Produto biológico,Produto combinado,Radiação,Suplemento alimentar,Teste diagnóstico
tipo_intervencao,,,,,,,,,,,
BEHAVIORAL,3107,0,0,0,0,0,0,0,0,0,0
BIOLOGICAL,0,0,0,0,0,0,63,0,0,0,0
COMBINATION_PRODUCT,0,0,0,0,0,0,0,52,0,0,0
DEVICE,0,683,0,0,0,0,0,0,0,0,0
DIAGNOSTIC_TEST,0,0,0,0,0,0,0,0,0,0,75
DIETARY_SUPPLEMENT,0,0,0,0,0,0,0,0,0,1141,0
DRUG,0,0,0,1790,0,0,0,0,0,0,0
GENETIC,0,0,13,0,0,0,0,0,0,0,0
OTHER,0,0,0,0,2173,0,0,0,0,0,0


- **Tipo de intervenção:** os 11 códigos foram traduzidos para português: Comportamental (3 107 estudos), Outro (2 173), Medicamento (1 790), Suplemento alimentar (1 141), Dispositivo médico (683), Procedimento (590), Teste de diagnóstico (75), Produto biológico (63), Produto combinado (52), Genético (13) e Radiação (10). Os tipos pequenos (Genético e Radiação) não foram juntos ao "Outro", por serem intervenções muito específicas. Como um estudo pode ter vários tipos, os números somam mais do que o total de estudos.

In [531]:
motivos = df_estudos["motivo_paragem"].dropna()
print(len(motivos), "motivos,", motivos.nunique(), "diferentes")
print(motivos.sample(30, random_state=1).to_string())

496 motivos, 461 diferentes
6015                                                In the early stage of the research, this research was limited by the application permission of research drugs and instruments, and the research will have to be interrupted, which is extremely regrettable.
4386                                                                                                                                                                                                                                           lack of follow up
6522                                                                                                                                                                                                                                         Limited recruitment
5624                                                                                                                                                                                                     

In [532]:
def categorizar_motivo(texto):
    if pd.isna(texto):
       return None
    t = " " + texto.lower() + " "
    if "covid" in t or "pandemic" in t or "lockdown" in t:
       return "COVID-19"
    elif "safety" in t or "adverse" in t or "futility" in t or "efficacy" in t or "risk" in t or "tolerab" in t or "interim" in t or "statistic" in t or "recall" in t or "contamina" in t or "transaminase" in t or "liver" in t or "toxic" in t:
       return  "Segurança e eficácia" 
    elif "fund" in t or "financ" in t or "budget" in t or "resource" in t or "staff" in t or "grant" in t:
        return "Financiamento e recursos"      
    elif "ethic" in t or "irb" in t or "committee" in t or "approv" in t or "regulat" in t or "authoriz" in t or "autoriz" in t:
       return  "Regulatório e ética"
    elif "logistic" in t or  "technical" in t or "device" in t or "pharmacy" in t or " ship" in t or "shipping" in t or "shipment" in t or "supply" in t or "study drug" in t or "administrat" in t:
       return "Logística e problemas técnicos"    
    elif  "recruit" in t or "enrol" in t or "accrual" in t or "inclusion" in t or "follow up" in t or "participant" in t or "patient" in t or "subject" in t or "eligib" in t or "drop" in t or "adherence" in t:
       return "Recrutamento e adesão"
    elif "protocol" in t or "amendment" in t or "design" in t:
       return "Alteração do protocolo"  
    elif  "sponsor" in t:
       return "Decisão do patrocinador"
    elif  "investigator" in t or " pi " in t or "team" in t:
       return "Investigador ou equipa"
    else:
        return "Outro"    

In [533]:
df_estudos["categoria_motivo"] = df_estudos["motivo_paragem"].apply(categorizar_motivo)

In [534]:
df_estudos["categoria_motivo"].value_counts(dropna=False)
#os 7935 são estudos que continua, por isso não se aplica um motivo

categoria_motivo
NaN                               7935
Recrutamento e adesão              144
Financiamento e recursos           108
COVID-19                            61
Outro                               57
Investigador ou equipa              42
Segurança e eficácia                39
Logística e problemas técnicos      20
Regulatório e ética                 11
Decisão do patrocinador              9
Alteração do protocolo               5
Name: count, dtype: int64

In [535]:
pd.set_option("display.max_colwidth", None)
outros = df_estudos.loc[df_estudos["categoria_motivo"] == "Outro", "motivo_paragem"]
print(outros.sample(40, random_state=1).to_string())

6796                                                                                                                               Collaborator discontinued at this site to pursue the study with a different site.
222                                                                                                                                                                                              Unproven hypothesis
4873                                                                                                                                                                 Study terminated for strategic business reasons
6444                                                                                                                                                                                    Revised development strategy
5276                                                                                                                                                

In [536]:
com_motivo = df_estudos[df_estudos["categoria_motivo"].notna()]
amostra = com_motivo.sample(30, random_state=2)[["motivo_paragem", "categoria_motivo"]]
print(amostra.to_string())

                                                                                                                                                                                                                                            motivo_paragem                categoria_motivo
6162                                                                                                                                                                                                                           no enough enrolled patients           Recrutamento e adesão
4494                                                                                                                                                                                                                  target number of patient not reached           Recrutamento e adesão
6146                                                                                                                                                   

In [537]:
df_estudos.loc[[8004, 2145, 3238, 5886], ["motivo_paragem", "categoria_motivo"]]

,motivo_paragem,categoria_motivo
8004,"Grant proposal not funded, study has been withdrawn with IRB",Financiamento e recursos
2145,MR protocol was not ready for the study. Grant for the study expired.,Financiamento e recursos
3238,The decision to terminate clinical development of lotiglipron is based on pharmacokinetic data from Phase 1 drug-drug-interaction studies and laboratory measurements of elevated transaminases in these Phase 1 studies as well as a Phase 2 study.,Segurança e eficácia
5886,Study product voluntarily recalled by sponsor. Listeria identified at the manufacturing facility (not in tested finished product).,Segurança e eficácia


In [538]:
parados = df_estudos["estado"].isin(["TERMINATED", "SUSPENDED", "WITHDRAWN"])
sem_motivo = parados & df_estudos["categoria_motivo"].isna()
sem_motivo.sum()

np.int64(40)

In [539]:
df_estudos.loc[sem_motivo, "categoria_motivo"] = "Motivo não indicado"

In [540]:
df_estudos["categoria_motivo"].value_counts(dropna=False)

categoria_motivo
NaN                               7895
Recrutamento e adesão              144
Financiamento e recursos           108
COVID-19                            61
Outro                               57
Investigador ou equipa              42
Motivo não indicado                 40
Segurança e eficácia                39
Logística e problemas técnicos      20
Regulatório e ética                 11
Decisão do patrocinador              9
Alteração do protocolo               5
Name: count, dtype: int64

In [541]:
df_estudos.loc[parados, "categoria_motivo"].isna().sum()       # parados SEM categoria

np.int64(0)

In [542]:
df_estudos.loc[~parados, "categoria_motivo"].notna().sum()     # não parados COM categoria

np.int64(0)

- **Motivo de paragem:** texto livre (496 motivos, 461 textos diferentes), categorizado com uma função de palavras-chave em inglês. Ordem de prioridade (as causas antes de quem decidiu): COVID-19, Segurança e eficácia, Financiamento e recursos, Regulatório e ética, Logística e problemas técnicos, Recrutamento e adesão, Alteração do protocolo, Decisão do patrocinador, Investigador ou equipa.
- **Validação:** numa amostra aleatória de 30 motivos, 25 (83,3 %) estavam certos; os erros encontrados foram corrigidos (palavras-chave novas e mudança de ordem).
- **Outro:** 57 motivos (11,5 % dos 496), sobretudo motivos vagos (ex.: "withdrawn", "never started").
- **Sem motivo:** os 40 estudos parados sem motivo ficaram como "Motivo não indicado"; os estudos não parados ficaram vazios (não se aplica).
- **Limitação:** o método de palavras-chave não deteta negações (ex.: "not for safety reasons") nem palavras que não foram previstas.